EXP4: CNN FOR IMAGE CLASSIFICATION

In [1]:
#Step01: Importing libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

#Step02: transform pipeline : preprocessing
transform = transforms.Compose([ #this function chains multiple image transformation
    #.totensor: convert pil image or array to torch, rearranges the dimensions which are originally(height,width,channels) to (channels,height,width),
    #scales picture value(0-255) to 0.0 to 1.0
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) #0.5 is the center value of(-1,1) so it helps to center around 0
])

#step 03: loading the dataset
#loading the ds cifar10 root = means storing data locally, training = true, download ds if not available and apply transform means preprocessing pipeline
train_dataset = torchvision.datasets.CIFAR10(root ='./data', train = True, download = True, transform = transform)
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle = True) #loads training ds each batch has 64 images and shuffles the data at each epoch

test_dataset = torchvision.datasets.CIFAR10(root = './data', train = False, download = True, transform = transform)
testloader = torch.utils.data.DataLoader(test_dataset, batch_size = 64, shuffle = False)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

100%|██████████| 170M/170M [02:16<00:00, 1.25MB/s]


In [2]:
#step04: cnn
class SimpleCNN(nn.Module):
  def __init__(self):#used to define all the layers it runs when model is defined
     super(SimpleCNN,self).__init__() #calls the constructor of parent class
     #conv layer 1: 3 input channels(RGB), 16 filters, 3 x 3 kernel size
     #Feature extraction part (Convolution + pooling)
     self.conv1 = nn.Conv2d(3, 16, kernel_size = 3, padding = 1) #conv layer 1
     #conv layer 2 : 16 input channels, 32 filters, 3 x 3 kernel size
     self.conv2 = nn.Conv2d(16, 32, kernel_size = 3, padding = 1) #conv layer 2
     #pooling layer
     self.pool = nn.MaxPool2d(2, 2) #reduces dimension by half (e.g., 32x32 -> 16x16) , performs max pooling with stride = 2 and kernel_size = 2 x 2

     #Fully connected layer(classifier)
     #After 2 pooling layer the dimensions reduce from 32 x 32 to 8 x 8 i.e 32filters x 8 x 8
     self.fc1 = nn.Linear(32 * 8 * 8, 128) #output has 128 neurons
     self.fc2 = nn.Linear(128, 10) #output is 10 that means 10 classes
     self.relu = nn.ReLU() #activation function

  def forward(self,x): #forward function
      x = self.pool(self.relu(self.conv1(x)))
      x = self.pool(self.relu(self.conv2(x)))
      x = x.view(-1, 32 * 8 * 8) #flatten converts 3d feature map into 1d vector and -1 means automatic calculate batchsize
      x = self.relu(self.fc1(x))
      x = self.fc2(x)
      return x

model = SimpleCNN()

In [3]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

for epoch in range(5):
  running_loss = 0.0
  for i,data in enumerate(trainloader,0):
    inputs,labels = data
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()

  print(f'Epoch {epoch + 1}, Loss: {running_loss / len(trainloader):.3f}')

print('Finished Training')


Epoch 1, Loss: 1.450
Epoch 2, Loss: 1.117
Epoch 3, Loss: 0.972
Epoch 4, Loss: 0.873
Epoch 5, Loss: 0.791
Finished Training
